# FAQ Retrieval Chatbot: implementation and evaluation
An undergraduate information-retrieval project using a fixed catalogue of 18 sample online-store FAQs. Compare a word TF-IDF baseline with a word/character hybrid, select rejection settings on a calibration set, and evaluate once on a separate set of paraphrases and unrelated questions.

**Scope:** retrieval of existing answers; no LLM, generated answers, or real payment processing. Policies are fictional demonstrations, not advice about an actual store.

**Provenance:** extends the uploaded `Internship_project_1(1).ipynb`. The original FAQ catalogue is retained. Retrieval, evaluation and documentation upgrades and manually authored evaluation examples were added with AI assistance. Describe your own contribution accurately in applications.

## Run instructions
Use Python 3.10+ in VS Code, Jupyter or Colab. Install the packages below if needed, then run all cells in order. No API key, paid service, NLTK resource download or external dataset is needed. Interactive chat is optional and disabled by default.


In [1]:
# Uncomment once if these packages are missing, then restart the kernel.
# %pip install numpy pandas scipy scikit-learn matplotlib
import re
import sys
import platform
from dataclasses import dataclass
import numpy as np
import pandas as pd
import sklearn
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
pd.set_option('display.max_colwidth', 90)
print({'python': platform.python_version(), 'scikit_learn': sklearn.__version__,
       'numpy': np.__version__, 'pandas': pd.__version__})


{'python': '3.12.14', 'scikit_learn': '1.8.0', 'numpy': '2.3.5', 'pandas': '2.2.3'}


## 1. Knowledge base
Each question has a stable ID. The vectorizers learn only from canonical FAQ questions; evaluation queries are never used to fit vocabulary or document frequencies. Add or revise policies here, then rebuild the retriever.


In [2]:
FAQS = [{'id': 'FAQ01', 'question': 'What payment methods do you accept?', 'answer': 'We accept credit/debit cards, bank transfers, and cash on delivery.'}, {'id': 'FAQ02', 'question': 'How can I track my order?', 'answer': "You can track your order from the 'My Orders' section using your order ID."}, {'id': 'FAQ03', 'question': 'What is your return policy?', 'answer': 'Products can be returned within 30 days if they are unused and in their original packaging.'}, {'id': 'FAQ04', 'question': 'How do I request a refund?', 'answer': "Go to your order history, select the order, and click 'Request Refund'."}, {'id': 'FAQ05', 'question': 'How long does delivery take?', 'answer': 'Standard delivery takes 3–7 business days, depending on your location.'}, {'id': 'FAQ06', 'question': 'Can I cancel my order?', 'answer': 'Yes, you can cancel your order before it has been shipped.'}, {'id': 'FAQ07', 'question': 'How do I reset my account password?', 'answer': "Click 'Forgot Password' on the login page and follow the instructions sent to your email."}, {'id': 'FAQ08', 'question': 'Do you offer international shipping?', 'answer': 'Yes, we ship to selected countries. Shipping charges vary by destination.'}, {'id': 'FAQ09', 'question': 'How can I contact customer support?', 'answer': 'You can contact us via email, live chat, or our customer support phone number.'}, {'id': 'FAQ10', 'question': 'Is cash on delivery available?', 'answer': 'Yes, cash on delivery is available for eligible locations.'}, {'id': 'FAQ11', 'question': 'How do I create an account?', 'answer': "Click 'Sign Up' on the homepage and fill in your details to create a new account."}, {'id': 'FAQ12', 'question': 'Is it safe to save my card details?', 'answer': 'Yes, we use industry-standard encryption to securely store your card details.'}, {'id': 'FAQ13', 'question': 'Can I change my delivery address after ordering?', 'answer': "You can change your delivery address before the order is shipped, from the 'My Orders' section."}, {'id': 'FAQ14', 'question': 'Do you charge extra for online payments?', 'answer': 'No, we do not charge any extra fees for online payments.'}, {'id': 'FAQ15', 'question': 'What should I do if my payment fails?', 'answer': 'Please check your card details and try again, or contact your bank if the issue persists.'}, {'id': 'FAQ16', 'question': 'How do I apply a discount coupon?', 'answer': "Enter your coupon code at checkout in the 'Apply Coupon' field before placing your order."}, {'id': 'FAQ17', 'question': 'Can I pay using a mobile wallet?', 'answer': 'Yes, we support payments through major mobile wallets.'}, {'id': 'FAQ18', 'question': 'How do I delete my account?', 'answer': "Go to Account Settings and select 'Delete Account' to permanently remove your account."}]
assert len({f['id'] for f in FAQS}) == len(FAQS)
pd.DataFrame(FAQS)


## 2. Retrieval and input handling
The baseline uses word unigrams. The hybrid combines word unigrams/bigrams and character n-grams, intended to handle local spelling variations. Both preserve words such as `not`: blanket stopword removal can lose meaning. Character overlap is not semantic understanding.

Reject a query when its top similarity is below a selected threshold or when the difference between the best two matches is too small. Scores are similarities, **not probabilities or confidence percentages**. The margin can reject some ambiguous matches, but cannot reliably detect unsupported policy details.


In [3]:
def clean_text(text):
    if not isinstance(text, str):
        raise TypeError('The query must be a string.')
    return re.sub(r'\s+', ' ', text.casefold()).strip()

@dataclass(frozen=True)
class RetrievalResult:
    accepted: bool
    faq_id: str | None
    answer: str
    score: float
    margin: float
    reason: str
    candidates: tuple

class FAQRetriever:
    def __init__(self, faqs, method='hybrid', word_weight=0.7):
        if method not in {'baseline', 'hybrid'}:
            raise ValueError('Unknown retrieval method.')
        if len(faqs) < 2 or len({f['id'] for f in faqs}) != len(faqs):
            raise ValueError('Use at least two FAQs with unique IDs.')
        if not 0 < word_weight < 1:
            raise ValueError('word_weight must lie between 0 and 1.')
        self.faqs, self.method, self.word_weight = list(faqs), method, word_weight
        corpus = [clean_text(f['question']) for f in faqs]
        self.word = TfidfVectorizer(ngram_range=(1, 1 if method == 'baseline' else 2),
                                    sublinear_tf=True)
        word_matrix = self.word.fit_transform(corpus)
        if method == 'hybrid':
            self.char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)
            char_matrix = self.char.fit_transform(corpus)
            self.matrix = self._combine(word_matrix, char_matrix)
        else:
            self.matrix = word_matrix

    def _combine(self, word, char):
        # Square-root weights give the desired relative contribution to dot products.
        return normalize(hstack([np.sqrt(self.word_weight)*word,
                                 np.sqrt(1-self.word_weight)*char]).tocsr())

    def rank(self, query):
        text = clean_text(query)
        if not re.search(r'[^\W_]', text, flags=re.UNICODE):
            return [], 'empty_or_punctuation'
        if len(text) > 1000:
            return [], 'input_too_long'
        word = self.word.transform([text])
        vector = self._combine(word, self.char.transform([text])) if self.method == 'hybrid' else word
        if vector.nnz == 0:
            return [], 'no_vocabulary_overlap'
        scores = cosine_similarity(vector, self.matrix).ravel()
        order = np.argsort(-scores, kind='stable')[:3]
        return [(self.faqs[i]['id'], float(scores[i]), self.faqs[i]['question']) for i in order], 'ranked'

    def predict(self, query, threshold=0.5, min_margin=0.0):
        candidates, reason = self.rank(query)
        fallback = 'I could not find a reliable FAQ match. Please rephrase or contact customer support.'
        if not candidates:
            return RetrievalResult(False, None, fallback, 0.0, 0.0, reason, ())
        score = candidates[0][1]
        margin = score-candidates[1][1]
        accepted = score >= threshold and margin >= min_margin
        faq_id = candidates[0][0] if accepted else None
        answer = next(f['answer'] for f in self.faqs if f['id']==faq_id) if accepted else fallback
        reason = 'matched' if accepted else ('low_similarity' if score < threshold else 'ambiguous_match')
        return RetrievalResult(accepted, faq_id, answer, score, margin, reason, tuple(candidates))

models = {name: FAQRetriever(FAQS, name) for name in ['baseline', 'hybrid']}


## 3. Evaluation data and experimental protocol
Two calibration paraphrases and two held-out paraphrases per FAQ, plus unrelated and unsupported questions. `expected_id=None` means the correct outcome is rejection. These examples were manually authored with AI assistance, not collected from real users; results on this small artificial benchmark do not establish production accuracy. Synonyms, negation, and unsupported details remain challenging.

Select thresholds and the final retrieval method using **calibration data only**. Report both methods on held-out data for transparency, but do not change settings after observing that table. A future revision requires fresh independently labelled test queries.


In [4]:
CALIBRATION = [{'query': 'Which ways can I pay?', 'expected_id': 'FAQ01'}, {'query': 'Are bank transfers accepted?', 'expected_id': 'FAQ01'}, {'query': 'Where is my order tracking?', 'expected_id': 'FAQ02'}, {'query': 'How do I check order status?', 'expected_id': 'FAQ02'}, {'query': 'Can I return an unused item?', 'expected_id': 'FAQ03'}, {'query': 'How many days do I have to return a product?', 'expected_id': 'FAQ03'}, {'query': 'How can I get my money refunded?', 'expected_id': 'FAQ04'}, {'query': 'Where do I request a refund?', 'expected_id': 'FAQ04'}, {'query': 'What is the delivery duration?', 'expected_id': 'FAQ05'}, {'query': 'How many days until delivery?', 'expected_id': 'FAQ05'}, {'query': 'Can I cancel an unshipped purchase?', 'expected_id': 'FAQ06'}, {'query': 'How do I cancel my order?', 'expected_id': 'FAQ06'}, {'query': 'I forgot my password, how do I reset it?', 'expected_id': 'FAQ07'}, {'query': 'How can I recover my account password?', 'expected_id': 'FAQ07'}, {'query': 'Do you ship orders abroad?', 'expected_id': 'FAQ08'}, {'query': 'Is international delivery offered?', 'expected_id': 'FAQ08'}, {'query': 'How do I reach customer support?', 'expected_id': 'FAQ09'}, {'query': 'How can I speak to your support team?', 'expected_id': 'FAQ09'}, {'query': 'Can I pay cash when my order arrives?', 'expected_id': 'FAQ10'}, {'query': 'Do you support cash on delivery?', 'expected_id': 'FAQ10'}, {'query': 'How do I register a new account?', 'expected_id': 'FAQ11'}, {'query': 'Where can I sign up?', 'expected_id': 'FAQ11'}, {'query': 'Are saved card details secure?', 'expected_id': 'FAQ12'}, {'query': 'Is storing my debit card safe?', 'expected_id': 'FAQ12'}, {'query': 'Can I update my delivery address?', 'expected_id': 'FAQ13'}, {'query': 'How do I change the shipping address?', 'expected_id': 'FAQ13'}, {'query': 'Are there extra online payment fees?', 'expected_id': 'FAQ14'}, {'query': 'Do online payments have additional charges?', 'expected_id': 'FAQ14'}, {'query': 'My payment failed, what should I do?', 'expected_id': 'FAQ15'}, {'query': 'How do I fix a failed payment?', 'expected_id': 'FAQ15'}, {'query': 'Where do I enter a discount coupon?', 'expected_id': 'FAQ16'}, {'query': 'How can I use a coupon code?', 'expected_id': 'FAQ16'}, {'query': 'Do you accept mobile wallet payments?', 'expected_id': 'FAQ17'}, {'query': 'Can I use a mobile wallet?', 'expected_id': 'FAQ17'}, {'query': 'How can I delete my account?', 'expected_id': 'FAQ18'}, {'query': 'Where is the account deletion option?', 'expected_id': 'FAQ18'}, {'query': 'What is the weather tomorrow?', 'expected_id': None}, {'query': 'Tell me a joke', 'expected_id': None}, {'query': 'How do I cook rice?', 'expected_id': None}, {'query': 'Who won the football match?', 'expected_id': None}, {'query': 'Can I reset my router password?', 'expected_id': None}, {'query': 'How do I cancel my gym membership?', 'expected_id': None}, {'query': 'Can I track a satellite?', 'expected_id': None}, {'query': 'Does this item have a lifetime warranty?', 'expected_id': None}, {'query': 'Can I pay with cryptocurrency?', 'expected_id': None}, {'query': 'What is my current bank balance?', 'expected_id': None}, {'query': 'Can I return a damaged item after a year?', 'expected_id': None}, {'query': 'Hello', 'expected_id': None}, {'query': '!!!', 'expected_id': None}, {'query': 'the and is', 'expected_id': None}, {'query': '', 'expected_id': None}]
TEST = [{'query': 'Can I purchase with a debit card?', 'expected_id': 'FAQ01'}, {'query': 'What forms of payment are supported?', 'expected_id': 'FAQ01'}, {'query': 'Where can I track my parcel?', 'expected_id': 'FAQ02'}, {'query': 'Show me the delivery progress of my purchase', 'expected_id': 'FAQ02'}, {'query': 'What are the conditions for returning goods?', 'expected_id': 'FAQ03'}, {'query': 'Is there a return window for purchases?', 'expected_id': 'FAQ03'}, {'query': 'I want to submit a refund request', 'expected_id': 'FAQ04'}, {'query': 'How do I get my payment back?', 'expected_id': 'FAQ04'}, {'query': 'When will my parcel arrive?', 'expected_id': 'FAQ05'}, {'query': 'What is the standard shipping time?', 'expected_id': 'FAQ05'}, {'query': 'I want to stop my order before dispatch', 'expected_id': 'FAQ06'}, {'query': 'Is cancelling a purchase possible?', 'expected_id': 'FAQ06'}, {'query': 'Help me change a forgotten password', 'expected_id': 'FAQ07'}, {'query': 'Where is the password reset option?', 'expected_id': 'FAQ07'}, {'query': 'Can I receive an order in another country?', 'expected_id': 'FAQ08'}, {'query': 'Do you deliver outside this country?', 'expected_id': 'FAQ08'}, {'query': 'Where can I contact customer service?', 'expected_id': 'FAQ09'}, {'query': 'I need to get in touch with the help team', 'expected_id': 'FAQ09'}, {'query': 'Is cash payment at delivery allowed?', 'expected_id': 'FAQ10'}, {'query': 'Can the courier collect cash for my purchase?', 'expected_id': 'FAQ10'}, {'query': 'I want to create a customer account', 'expected_id': 'FAQ11'}, {'query': 'How do I join as a new customer?', 'expected_id': 'FAQ11'}, {'query': 'How are my stored payment card details protected?', 'expected_id': 'FAQ12'}, {'query': 'Can I safely save my credit card information?', 'expected_id': 'FAQ12'}, {'query': 'I entered the wrong address for delivery', 'expected_id': 'FAQ13'}, {'query': 'Can my parcel be sent to a different address?', 'expected_id': 'FAQ13'}, {'query': 'Will paying online cost extra?', 'expected_id': 'FAQ14'}, {'query': 'Is there a surcharge for online payment?', 'expected_id': 'FAQ14'}, {'query': 'My card transaction was unsuccessful', 'expected_id': 'FAQ15'}, {'query': 'The payment did not go through', 'expected_id': 'FAQ15'}, {'query': 'Can I apply a promo code at checkout?', 'expected_id': 'FAQ16'}, {'query': 'How do I redeem my discount voucher?', 'expected_id': 'FAQ16'}, {'query': 'Is paying through a digital wallet possible?', 'expected_id': 'FAQ17'}, {'query': 'Which mobile wallets can I pay with?', 'expected_id': 'FAQ17'}, {'query': 'I want to remove my customer account', 'expected_id': 'FAQ18'}, {'query': 'How do I permanently close my account?', 'expected_id': 'FAQ18'}, {'query': 'Explain quantum physics', 'expected_id': None}, {'query': 'Translate good morning into Japanese', 'expected_id': None}, {'query': 'What is the capital of Japan?', 'expected_id': None}, {'query': 'Can I refund my train ticket?', 'expected_id': None}, {'query': 'How do I delete my phone photos?', 'expected_id': None}, {'query': 'How do I track my fitness progress?', 'expected_id': None}, {'query': 'Can I change my email address?', 'expected_id': None}, {'query': 'Do you offer student discounts?', 'expected_id': None}, {'query': 'What is my order number?', 'expected_id': None}, {'query': 'Do you ship frozen food?', 'expected_id': None}, {'query': 'How do I reset my WiFi password?', 'expected_id': None}, {'query': 'Can I pay in monthly instalments?', 'expected_id': None}, {'query': 'Hi there', 'expected_id': None}, {'query': '???', 'expected_id': None}, {'query': 'and the or', 'expected_id': None}, {'query': '', 'expected_id': None}]
assert not ({r['query'] for r in CALIBRATION if r['query'].strip()} &
            {r['query'] for r in TEST if r['query'].strip()})
for label, rows in [('calibration', CALIBRATION), ('held_out', TEST)]:
    print(label, 'total:', len(rows), 'answerable:', sum(r['expected_id'] is not None for r in rows),
          'reject:', sum(r['expected_id'] is None for r in rows))


calibration total: 51 answerable: 36 reject: 15
held_out total: 52 answerable: 36 reject: 16


In [5]:
def evaluate(model, examples, threshold, min_margin):
    records = []
    for example in examples:
        result = model.predict(example['query'], threshold, min_margin)
        expected = example['expected_id']
        records.append({'query': example['query'], 'expected_id': expected,
                        'predicted_id': result.faq_id, 'accepted': result.accepted,
                        'score': result.score, 'margin': result.margin,
                        'reason': result.reason, 'correct': result.faq_id == expected,
                        'top_id': result.candidates[0][0] if result.candidates else None})
    frame = pd.DataFrame(records)
    answerable = frame['expected_id'].notna()
    correct_match = answerable & frame['correct']
    n_pos, n_neg = int(answerable.sum()), int((~answerable).sum())
    match_recall = correct_match.sum()/n_pos if n_pos else 0.0
    rejection_rate = ((~answerable) & (~frame['accepted'])).sum()/n_neg if n_neg else 0.0
    summary = {
        'overall_accuracy': float(frame['correct'].mean()),
        'answerable_correct_rate': float(match_recall),
        'unrelated_rejection_rate': float(rejection_rate),
        'balanced_correct_rate': float((match_recall+rejection_rate)/2),
        'accepted_answer_precision': float(correct_match.sum()/frame['accepted'].sum()) if frame['accepted'].sum() else 0.0,
        'coverage': float(frame['accepted'].mean()),
        'answerable_top1_before_rejection': float((frame.loc[answerable,'top_id']==frame.loc[answerable,'expected_id']).mean()),
    }
    return summary, frame

# Threshold search is confined to calibration queries; cache rankings to avoid recomputation.
def choose_settings(model):
    rankings = [model.rank(r['query'])[0] for r in CALIBRATION]
    positive = np.array([r['expected_id'] is not None for r in CALIBRATION])
    expected = [r['expected_id'] for r in CALIBRATION]
    trials = []
    for threshold in np.arange(0.15, 0.81, 0.05):
        for margin in [0.0, 0.05, 0.10, 0.15, 0.20]:
            accepted = np.array([bool(c) and c[0][1]>=threshold and c[0][1]-c[1][1]>=margin for c in rankings])
            correct = np.array([accepted[i] and c[0][0]==expected[i] for i,c in enumerate(rankings)])
            recall = correct[positive].mean()
            rejection = (~accepted[~positive]).mean()
            precision = correct.sum()/accepted.sum() if accepted.sum() else 0.0
            trials.append({'threshold': float(threshold), 'min_margin': margin,
                           'balanced_correct_rate': (recall+rejection)/2,
                           'accepted_answer_precision': precision})
    table = pd.DataFrame(trials).sort_values(
        ['balanced_correct_rate','accepted_answer_precision','threshold','min_margin'],
        ascending=[False,False,True,True], kind='stable')
    return table.iloc[0].to_dict(), table

settings = {}
for name, model in models.items():
    settings[name], _ = choose_settings(model)
calibration_table = pd.DataFrame(settings).T
selected_name = calibration_table.sort_values(
    ['balanced_correct_rate','accepted_answer_precision'], ascending=False, kind='stable').index[0]
print('Selected using calibration only:', selected_name)
print(calibration_table.round(4).to_string())


Selected using calibration only: hybrid
          threshold  min_margin  balanced_correct_rate  accepted_answer_precision
baseline       0.65        0.10                 0.5861                     0.8462
hybrid         0.15        0.05                 0.6417                     0.7297


## 4. Held-out results
Balanced correct rate averages the answerable correct-match rate and unrelated rejection rate. Accepted-answer precision counts wrong-topic answers as errors as well as answers to unrelated queries. Coverage is the fraction of all queries answered. Rejecting everything cannot obtain a perfect balanced score.


In [6]:
test_tables, summaries = {}, {}
for name, model in models.items():
    config = settings[name]
    summaries[name], test_tables[name] = evaluate(model, TEST, config['threshold'], config['min_margin'])
results = pd.DataFrame(summaries).T
print(results.round(4).to_string())
print('Final model remains:', selected_name)


          overall_accuracy  answerable_correct_rate  unrelated_rejection_rate  balanced_correct_rate  accepted_answer_precision  coverage  answerable_top1_before_rejection
baseline            0.3269                   0.1667                    0.6875                 0.4271                     0.5455    0.2115                            0.6667
hybrid              0.5000                   0.5278                    0.4375                 0.4826                     0.5278    0.6923                            0.6667
Final model remains: hybrid


## 5. Error analysis
Inspect every failed query, not just a headline score. A related question with an unsupported detail may receive a plausible but incorrect answer. In real deployment, restrict the domain, review policies, and refer uncertain cases to a human. Do not describe this system as a semantic reasoning model.


In [7]:
failures = test_tables[selected_name].loc[lambda d: ~d['correct']]
print('Held-out errors:', len(failures))
print(failures[['query','expected_id','predicted_id','score','margin','reason']].round(3).to_string(index=False))
print('\nConclusion: these results apply only to the included manually authored benchmark.')
print('Next experiment: collect new independently labelled paraphrases, typos, negations and ambiguous queries;')
print('compare sentence embeddings with this baseline using a new untouched test set.')


Held-out errors: 26
                                        query expected_id predicted_id  score  margin          reason
            Can I purchase with a debit card?       FAQ01        FAQ12  0.233   0.063         matched
         What forms of payment are supported?       FAQ01         None  0.388   0.021 ambiguous_match
 Show me the delivery progress of my purchase       FAQ02        FAQ13  0.336   0.057         matched
 What are the conditions for returning goods?       FAQ03         None  0.249   0.045 ambiguous_match
                How do I get my payment back?       FAQ04        FAQ15  0.419   0.150         matched
                  When will my parcel arrive?       FAQ05        FAQ06  0.264   0.074         matched
          What is the standard shipping time?       FAQ05        FAQ03  0.362   0.110         matched
      I want to stop my order before dispatch       FAQ06         None  0.427   0.028 ambiguous_match
           Is cancelling a purchase possible?       FAQ06     

## 6. Behaviour checks and demonstration
Checks cover exact FAQ retrieval, invalid inputs and deterministic ranking. They validate implementation behaviour, not real-world accuracy.


In [8]:
for model in models.values():
    for faq in FAQS:
        assert model.predict(faq['question'], 0.5, 0).faq_id == faq['id']
    for query in ['', '  ', '!!!', 'x'*1001]:
        assert not model.predict(query).accepted
    try:
        model.predict(None)
        raise AssertionError('Non-string input was accepted')
    except TypeError:
        pass
    assert model.rank('How do I reset my password?') == model.rank('How do I reset my password?')
print('All behaviour checks passed.')

bot = models[selected_name]
config = settings[selected_name]
def ask(query):
    return bot.predict(query, config['threshold'], config['min_margin'])

for query in ['How do I reset my password?', 'What is the weather tomorrow?', '']:
    reply = ask(query)
    print(f'\nQuery: {query!r}\nAnswer: {reply.answer}\nSimilarity: {reply.score:.3f}; reason: {reply.reason}')


All behaviour checks passed.

Query: 'How do I reset my password?'
Answer: Click 'Forgot Password' on the login page and follow the instructions sent to your email.
Similarity: 0.847; reason: matched

Query: 'What is the weather tomorrow?'
Answer: Products can be returned within 30 days if they are unused and in their original packaging.
Similarity: 0.462; reason: matched

Query: ''
Answer: I could not find a reliable FAQ match. Please rephrase or contact customer support.
Similarity: 0.000; reason: empty_or_punctuation


## 7. Optional interactive chat
Set `RUN_CHAT=True` and run this cell manually. Keep it False for Run All and automated execution. Type `exit` or `quit` to finish. The bot does not store conversations.


In [9]:
RUN_CHAT = False
if RUN_CHAT:
    print('FAQ assistant: ask about the sample store. Type exit to finish.')
    while True:
        try:
            query = input('You: ')
        except (EOFError, KeyboardInterrupt):
            print('\nSession ended.')
            break
        if clean_text(query) in {'exit', 'quit'}:
            print('Goodbye.')
            break
        if clean_text(query) in {'hello', 'hi', 'hey'}:
            print('Bot: Hello. What would you like to know about the store?')
            continue
        print('Bot:', ask(query).answer)


## References and reporting
- [scikit-learn: TF-IDF text feature extraction](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)
- [scikit-learn: cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)
- [scikit-learn: model selection and evaluation](https://scikit-learn.org/stable/model_selection.html)

